# 01 — Real-window prediction showcase

Visualizes a **single real window**: 4 omics input tracks + DNA one-hot strip +
true Hi-C + predicted Hi-C + difference map.

- Workflow source: `vis_gm12878.ipynb` (main-model prediction visualization;
  random window selection replaced by **median-PCC typical-sample selection**);
- Model: `MultiModalSeq2HiCModel` (weights, 512 bins);
- QC reference: reproduced on the same region (chr7 6.40-6.66 Mb, bins 128:256 of window chr7_6144000_7168000);
- All outputs are saved next to this notebook.


In [1]:
import os, sys, glob, pickle, csv, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import Dataset

# ---- path config (relative to this notebook; kernel cwd = notebook dir) ----
BASE = Path(os.getcwd()).resolve()
# All inputs are read-only NFS paths (never written to; no dependency on the local Mamba package)
WEIGHTS = (BASE / '..' / 'checkpoints' / 'main' / 'best_model.pth').resolve()
SCAN_DIR = Path('/mnt/nfs/clgou/Mamba/datasets/preprocessed_data_omic_512')                                 # all chr7 windows (NFS, read-only)
REF_PKL = (BASE / '..' / 'data' / 'chr7_6144000_7168000.pkl').resolve()   # reference-region window (local data/)
OUT_DIR = BASE                                             # all outputs are saved next to this notebook
os.makedirs(OUT_DIR, exist_ok=True)

# ---- mamba_ssm: compiled version preferred; otherwise pure-Python fallback in the package ----
try:
    from mamba_ssm import Mamba
except ImportError:
    sys.path.insert(0, str((BASE / '..' / 'src').resolve()))
    from mamba_ssm import Mamba

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device   :', device)
print('weights  :', WEIGHTS, WEIGHTS.exists())
print('scan_dir :', SCAN_DIR, SCAN_DIR.exists())
print('ref_pkl  :', REF_PKL, REF_PKL.exists())


device   : cuda
weights  : /home/common/yfzhu/selected codes/checkpoints/main/best_model.pth True
scan_dir : /mnt/nfs/clgou/Mamba/datasets/preprocessed_data_omic_512 True
ref_pkl  : /home/common/yfzhu/selected codes/data/chr7_6144000_7168000.pkl True


In [2]:
class DownsampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super(DownsampleBlock, self).__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class SequenceMambaEncoder(nn.Module):
    def __init__(self, in_channel=4, output_size=128):
        super(SequenceMambaEncoder, self).__init__()
        self.downsampler = nn.Sequential(
            DownsampleBlock(in_channel, 48, kernel_size=5, stride=5),
            DownsampleBlock(48, 96, kernel_size=5, stride=5),
            DownsampleBlock(96, 128, kernel_size=5, stride=5),
        )
        self.mamba_fw = Mamba(d_model=128, d_state=16, d_conv=4, expand=2)
        self.mamba_bw = Mamba(d_model=128, d_state=16, d_conv=4, expand=2)
        self.pooler = nn.AdaptiveAvgPool1d(512)
        self.projection = nn.Conv1d(128, output_size, kernel_size=1)
    def forward(self, x):
        x = self.downsampler(x); x = x.transpose(1, 2)
        x_fw = self.mamba_fw(x)
        x_rev = torch.flip(x, dims=[1]); x_bw = self.mamba_bw(x_rev); x_bw = torch.flip(x_bw, dims=[1])
        x = x_fw + x_bw
        x = x.transpose(1, 2); x = self.pooler(x); x = self.projection(x)
        return x

class OmicsMambaEncoder(nn.Module):
    def __init__(self, num_omics_features=4, d_model=128):
        super(OmicsMambaEncoder, self).__init__()
        self.projection = nn.Conv1d(num_omics_features, d_model, kernel_size=1)
        self.norm = nn.LayerNorm(d_model)
        self.mamba_fw = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
        self.mamba_bw = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
    def forward(self, x):
        x = self.projection(x); x = x.transpose(1, 2); x = self.norm(x)
        x_fw = self.mamba_fw(x)
        x_rev = torch.flip(x, dims=[1]); x_bw = self.mamba_bw(x_rev); x_bw = torch.flip(x_bw, dims=[1])
        x = x_fw + x_bw
        x = x.transpose(1, 2)
        return x

class MultiModalEncoder(nn.Module):
    def __init__(self, seq_in_channels=4, num_omics_features=4, encoder_out_dim=128):
        super(MultiModalEncoder, self).__init__()
        self.seq_encoder = SequenceMambaEncoder(in_channel=seq_in_channels, output_size=encoder_out_dim)
        self.omics_encoder = OmicsMambaEncoder(num_omics_features=num_omics_features, d_model=encoder_out_dim)
        self.fused_channels = encoder_out_dim
    def forward(self, seq_input, omics_input):
        seq_embedding = self.seq_encoder(seq_input)
        omics_embedding = self.omics_encoder(omics_input)
        fused_embedding = seq_embedding + omics_embedding
        return fused_embedding

class ResBlockDilated(nn.Module):
    def __init__(self, size, hidden=64, stride=1, dil=2):
        super(ResBlockDilated, self).__init__()
        kernel_size = size if size % 2 != 0 else size + 1
        pad_len = dil * (kernel_size // 2)
        self.res = nn.Sequential(nn.Conv2d(hidden, hidden, kernel_size, padding=pad_len, dilation=dil), nn.BatchNorm2d(hidden), nn.ReLU(), nn.Conv2d(hidden, hidden, kernel_size, padding=pad_len, dilation=dil), nn.BatchNorm2d(hidden),)
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.res(x) + x)

class Decoder(nn.Module):
    def __init__(self, in_channel, hidden=256, filter_size=3, num_blocks=5):
        super(Decoder, self).__init__()
        self.filter_size = filter_size
        self.conv_start = nn.Sequential(nn.Conv2d(in_channel, hidden, 3, 1, 1), nn.BatchNorm2d(hidden), nn.ReLU(),)
        blocks = [ResBlockDilated(self.filter_size, hidden=hidden, dil=2**(i+1)) for i in range(num_blocks)]
        self.res_blocks = nn.Sequential(*blocks)
        self.conv_end = nn.Conv2d(hidden, 1, 1)
    def forward(self, x):
        return self.conv_end(self.res_blocks(self.conv_start(x)))

class MultiModalSeq2HiCModel(nn.Module):
    def __init__(self, seq_in_channels=4, num_omics_features=4, encoder_out_dim=128):
        super(MultiModalSeq2HiCModel, self).__init__()
        self.encoder = MultiModalEncoder(seq_in_channels=seq_in_channels, num_omics_features=num_omics_features, encoder_out_dim=encoder_out_dim)
        decoder_in_channels = encoder_out_dim * 2
        self.decoder = Decoder(in_channel=decoder_in_channels, hidden=decoder_in_channels)
    def diagonalize(self, x):
        B, C, L = x.shape
        x_i = x.unsqueeze(3).repeat(1, 1, 1, L)
        x_j = x.unsqueeze(2).repeat(1, 1, L, 1)
        return torch.cat([x_i, x_j], dim=1)
    def forward(self, seq_one_hot, omics_signals):
        encoded_features = self.encoder(seq_one_hot, omics_signals)
        feature_map = self.diagonalize(encoded_features)
        pred_hic = self.decoder(feature_map)
        return pred_hic.squeeze(1)


In [3]:
class HiCDataAndOmicsDataset(Dataset):
    def __init__(self, pkl_files):
        self.file_paths = pkl_files
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        with open(self.file_paths[idx], 'rb') as f:
            data = pickle.load(f)
        sequence = torch.from_numpy(data['sequence_one_hot'].astype(np.float32))
        hic_matrix = torch.log2(torch.from_numpy(data['hic_matrix'].astype(np.float32)) + 1)
        omics_signals = torch.from_numpy(data['omics_signals'].astype(np.float32))
        return sequence, omics_signals, hic_matrix


def predict(model, seq, omics):
    with torch.no_grad():
        pred = model(seq.unsqueeze(0).to(device), omics.unsqueeze(0).to(device))
    return pred.squeeze(0).cpu().numpy()


def pcc(a, b):
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    b = np.asarray(b, dtype=np.float64).reshape(-1)
    if a.std() == 0 or b.std() == 0:
        return float('nan')
    return float(np.corrcoef(a, b)[0, 1])


In [5]:
OMICS_NAMES = ['CTCF', 'DNase', 'H3K27ac', 'H3K4me3']
OMICS_COLORS = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']
BASE_COLORS = {'A': '#2ecc71', 'C': '#3498db', 'G': '#e67e22', 'T': '#e74c3c'}


def make_panel_figure(title, seq_region, omics_region, real, pred, out_png):
    """Panel figure (same layout as the project's reference figure):
    4 omics signal tracks + DNA one-hot strip + true/predicted Hi-C side by side + difference + colorbar.
    real/pred are in log2 space; the color ceiling is the 99th percentile of positive true Hi-C."""
    vmax = np.percentile(real[real > 0], 99) if (real > 0).any() else 1.0
    diff = pred - real
    dlim = max(abs(float(diff.min())), abs(float(diff.max())), 1e-6)

    fig = plt.figure(figsize=(7, 20))
    gs = gridspec.GridSpec(8, 2, height_ratios=[1.0, 1.0, 1.0, 1.0, 0.9, 1.887, 3.77, 0.22],
                           width_ratios=[1, 1], hspace=0.22, wspace=0.08)
    fig.suptitle(title, fontsize=15, fontweight='bold')

    # --- 4 omics signal tracks ---
    track_axes = []
    for i in range(4):
        ax = fig.add_subplot(gs[i, :])
        sig = omics_region[i]
        ax.fill_between(np.arange(len(sig)), sig, color=OMICS_COLORS[i], alpha=0.35)
        ax.plot(sig, color=OMICS_COLORS[i], linewidth=1.2)
        ax.set_ylabel(OMICS_NAMES[i], fontsize=9, rotation=0, labelpad=28)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ['top', 'right', 'bottom']:
            ax.spines[s].set_visible(False)
        ax.set_xlim(0, len(sig) - 1)
        track_axes.append(ax)

    # --- DNA one-hot strip ---
    seq_region = np.asarray(seq_region)
    if seq_region.ndim == 2:  # (4, L) one-hot
        idx = np.argmax(seq_region, axis=0)
        letters = np.array(list('ACGT'))
        base_ids = idx
        nuc_colors = [BASE_COLORS[letters[b]] for b in base_ids]
        rgb = np.array([matplotlib.colors.to_rgb(c) for c in nuc_colors])
        strip = np.repeat(rgb[:, None, :], 3, axis=1)
        ax_dna = fig.add_subplot(gs[4, :])
        ax_dna.imshow(strip.transpose(1, 0, 2), aspect='auto')
        ax_dna.set_ylabel('DNA', fontsize=9, rotation=0, labelpad=28)
        ax_dna.set_xticks([]); ax_dna.set_yticks([])
        for s in ['top', 'right', 'bottom', 'left']:
            ax_dna.spines[s].set_visible(False)
    else:
        ax_dna = fig.add_subplot(gs[4, :]); ax_dna.axis('off')

    # --- true / predicted Hi-C side by side ---
    ax_real = fig.add_subplot(gs[5, 0])
    ax_real.imshow(real, cmap='Reds', vmin=0, vmax=vmax)
    ax_real.set_title('True Hi-C', fontweight='bold')
    ax_real.axis('off')

    ax_pred = fig.add_subplot(gs[5, 1])
    ax_pred.imshow(pred, cmap='Reds', vmin=0, vmax=vmax)
    ax_pred.set_title('Predicted Hi-C', fontweight='bold')
    ax_pred.axis('off')

    # --- difference map ---
    ax_diff = fig.add_subplot(gs[6, :])
    im_diff = ax_diff.imshow(diff, cmap='coolwarm', vmin=-dlim, vmax=dlim)
    ax_diff.set_title('Difference (Predicted - True)', fontweight='bold')
    ax_diff.axis('off')

    # --- colorbar ---
    cax = fig.add_subplot(gs[7, :])
    fig.colorbar(im_diff, cax=cax, orientation='horizontal')
    cax.tick_params(labelsize=8)

    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print('saved:', out_png)


In [ ]:
# ---- QC: reproduce the reference region (chr7 6.40-6.66 Mb) ----
# reference window chr7_6144000_7168000 (1 Mb, 2 kb/512 bins); region 6.40-6.656 Mb = bins 128:256,
# i.e. bp 256000:512000.
seq, omics, hic = HiCDataAndOmicsDataset([str(REF_PKL)])[0]
pred = predict(model, seq, omics)

crop = slice(128, 256)
real_crop = hic.numpy()[crop, crop]
pred_crop = pred[crop, crop]
omics_crop = omics.numpy()[:, crop]
seq_crop = seq.numpy()[:, 256000:512000]

print(f'reference region (bins 128:256, 6.40-6.656 Mb) cropped PCC = {pcc(pred_crop, real_crop):.4f}')
make_panel_figure(
    title='GM12878\nchr7: 6.40MB - 6.66MB',
    seq_region=seq_crop,
    omics_region=omics_crop,
    real=real_crop,
    pred=pred_crop,
    out_png=OUT_DIR / '01_reference_region_chr7_6400000_6656000.png',
)
# persist value: reference-region cropped PCC
with open(OUT_DIR / '01_reference_region_metrics.json', 'w') as fh:
    json.dump({'window': 'chr7_6144000_7168000', 'region_bins': '128:256',
               'region_bp': '6.40MB-6.656MB', 'crop_pcc': float(pcc(pred_crop, real_crop))}, fh, indent=2)
print('saved: 01_reference_region_metrics.json')


reference region (bins 128:256, 6.40-6.656 Mb) cropped PCC = 0.8030


saved: /home/common/yfzhu/selected codes/01_real_window_prediction/01_reference_region_chr7_6400000_6656000.png
saved: 01_reference_region_metrics.json
